# 06. 유사 질문 매칭 실험 (TF-IDF vs Sentence-BERT)

**목표**: 두 매칭 방법의 성능을 비교하여 의미 기반 검색의 실질적 우위를 검증

| 방법 | 설명 |
|------|------|
| TF-IDF + 코사인 유사도 | 토큰 빈도 기반, 형태소 분석 결과 활용 |
| Sentence-BERT | 한국어 문장 임베딩, 의미적 유사도 |

> **교수 중점사항**: TF-IDF 한계 인식 + BERT 비교가 필수 반영 항목  
> **평가 기준**: `data/splits/ground_truth.csv` (05_ground_truth.ipynb에서 생성)

## 0. 환경 설정

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

from utils.config import DATA_PROCESSED, PROJECT_ROOT, is_s3, S3_BUCKET, AWS_REGION

PREPROCESSED_PATH = DATA_PROCESSED / 'corpus_preprocessed.csv'
GROUND_TRUTH_PATH = PROJECT_ROOT / 'data/splits/ground_truth.csv'
EMBEDDINGS_DIR    = DATA_PROCESSED / 'embeddings'
EMBEDDINGS_DIR.mkdir(exist_ok=True)

# 한국어 Sentence-BERT 모델
SBERT_MODEL = 'jhgan/ko-sroberta-multitask'

print("환경 설정 완료")

## 1. 데이터 로드

In [ ]:
df       = pd.read_csv(PREPROCESSED_PATH)
df_gt    = pd.read_csv(GROUND_TRUTH_PATH)

# 매칭 DB: train split
df_db    = df[df['split'] == 'train'].reset_index(drop=True)

print(f"매칭 DB: {len(df_db):,}개")
print(f"Ground Truth 쿼리: {len(df_gt)}개")
df_gt.head(3)

## 2. TF-IDF 매칭

In [ ]:
# TF-IDF 벡터화 — 전처리된 토큰 사용
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_db['input_tokens'].fillna(''))

print(f"TF-IDF 행렬: {tfidf_matrix.shape}")


def match_tfidf(query: str, top_k: int = 5) -> list[int]:
    """쿼리를 TF-IDF 벡터로 변환 후 코사인 유사도 기준 상위 top_k 인덱스 반환."""
    query_vec = vectorizer.transform([query])
    scores    = cosine_similarity(query_vec, tfidf_matrix).flatten()
    return scores.argsort()[::-1][:top_k].tolist()


# Ground Truth 쿼리에 대해 TF-IDF 매칭 실행
tfidf_results = []
for _, row in tqdm(df_gt.iterrows(), total=len(df_gt), desc="TF-IDF 매칭"):
    top5 = match_tfidf(row['query'], top_k=5)
    tfidf_results.append(top5)

print("TF-IDF 매칭 완료")

## 3. Sentence-BERT 임베딩 생성 및 매칭

> **메모리 주의**: 로컬 실행 가능하나 느릴 수 있음.  
> 대용량 처리 시 SageMaker 사용 권장 (`docs/aws_migration.md Step 4` 참고)

In [ ]:
EMBED_PATH = EMBEDDINGS_DIR / 'db_embeddings.npy'

# 임베딩이 이미 있으면 로드, 없으면 생성 (재실행 시 시간 절약)
if EMBED_PATH.exists():
    db_embeddings = np.load(EMBED_PATH)
    print(f"임베딩 로드 완료: {db_embeddings.shape}")
else:
    print(f"임베딩 생성 중 (모델: {SBERT_MODEL})...")
    sbert_model   = SentenceTransformer(SBERT_MODEL)
    db_embeddings = sbert_model.encode(
        df_db['input_normalized'].fillna('').tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True  # 코사인 유사도 최적화
    )
    np.save(EMBED_PATH, db_embeddings)
    print(f"임베딩 저장 완료: {EMBED_PATH}")

# AWS 마이그레이션 후 S3 업로드 (docs/aws_migration.md Step 4 참고)
# if is_s3():
#     import boto3
#     s3 = boto3.client('s3', region_name=AWS_REGION)
#     s3.upload_file(str(EMBED_PATH), S3_BUCKET, 'embeddings/db_embeddings.npy')
#     print("S3 임베딩 업로드 완료")

In [ ]:
# 쿼리 임베딩 생성
if 'sbert_model' not in dir():
    sbert_model = SentenceTransformer(SBERT_MODEL)

query_embeddings = sbert_model.encode(
    df_gt['query'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)


def match_sbert(query_emb: np.ndarray, top_k: int = 5) -> list[int]:
    """임베딩 코사인 유사도 기준 상위 top_k 인덱스 반환."""
    scores = (db_embeddings @ query_emb).flatten()
    return scores.argsort()[::-1][:top_k].tolist()


sbert_results = [
    match_sbert(query_embeddings[i])
    for i in tqdm(range(len(df_gt)), desc="BERT 매칭")
]
print("Sentence-BERT 매칭 완료")

## 4. 매칭 결과 샘플 확인

In [ ]:
# 쿼리 1개에 대한 TF-IDF vs BERT 결과 비교
idx = 0
row = df_gt.iloc[idx]

print(f"[쿼리] {row['query'][:150]}")
print(f"[정답 idx] {row['correct_idx']}")
print()

print("=== TF-IDF Top 3 ===")
for rank, db_idx in enumerate(tfidf_results[idx][:3], 1):
    match_text = df_db.iloc[db_idx]['input'][:100]
    is_correct = "✅" if db_idx == row['correct_idx'] else "  "
    print(f"{is_correct} {rank}위 [{db_idx}]: {match_text}")

print()
print("=== Sentence-BERT Top 3 ===")
for rank, db_idx in enumerate(sbert_results[idx][:3], 1):
    match_text = df_db.iloc[db_idx]['input'][:100]
    is_correct = "✅" if db_idx == row['correct_idx'] else "  "
    print(f"{is_correct} {rank}위 [{db_idx}]: {match_text}")

## 5. 결과 저장

In [ ]:
RESULTS_PATH = DATA_PROCESSED / 'matching_results.csv'

df_gt['tfidf_top5'] = [str(r) for r in tfidf_results]
df_gt['sbert_top5'] = [str(r) for r in sbert_results]
df_gt.to_csv(RESULTS_PATH, index=False, encoding='utf-8-sig')

print(f"저장 완료: {RESULTS_PATH}")

# AWS 마이그레이션 후 S3 업로드 (docs/aws_migration.md Step 3 참고)
# if is_s3():
#     import boto3
#     s3 = boto3.client('s3', region_name=AWS_REGION)
#     s3.upload_file(str(RESULTS_PATH), S3_BUCKET, 'processed/matching_results.csv')
#     print("S3 업로드 완료")

## 6. 요약

| 항목 | 내용 |
|------|------|
| 매칭 DB | train split |
| 평가 쿼리 | 50개 (ground_truth.csv) |
| TF-IDF 벡터 | 형태소 분석 토큰 기반 |
| BERT 모델 | `jhgan/ko-sroberta-multitask` |
| 저장 파일 | `data/processed/matching_results.csv`, `data/processed/embeddings/db_embeddings.npy` |

> **다음 단계**: `07_evaluation.ipynb` — Top-1 / Top-3 / MRR 성능 비교 및 시각화